# Extraer `Planilha2` y `Planilha3` de `Paletização.xlsx`
### KinAnalytics

**Contexto (importante):** a diferencia de `Base 2.xlsb`, este archivo **no** trae
datos crudos para *reconstruir*. Verificado en el ZIP interno del `.xlsx`:

- Las 16 tablas dinámicas de `Planilha2` se alimentan de **Power Query externo**
  (consultas `DEPARA` y `Pre venda paletização`) → la fuente **no está** en el archivo.
- Las filas de la hoja por debajo de la fila ~81 están **vacías** (solo formato).
- Es **otro universo** de datos: total = **73.981.990** cajas (vs 5.45 M de `base`).

Por eso aquí **extraemos** (no reconstruimos) el contenido ya calculado:
- `Planilha3` → **maestro de Key Accounts** (dato real, 1.105 filas).
- `Planilha2` → las **16 dinámicas** (`Soma de VOL_CF` por `STATUS_PALLET_LASTRO`).

## Paso 1 · Abrir el archivo y preparar helpers

In [1]:
from pathlib import Path
import glob
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils.cell import range_boundaries

FN = [f for f in glob.glob(str(Path.cwd()/"*.xlsx")) if "alet" in Path(f).name.lower()][0]
print("Archivo:", Path(FN).name)
wb = load_workbook(FN, read_only=True, data_only=True)   # data_only = valores ya calculados
print("Hojas:", wb.sheetnames)

Archivo: Paletização.xlsx
Hojas: ['Planilha2', 'Planilha3']


## Paso 2 · `Planilha3` — Maestro de Key Accounts
Tabla dimensión: cada cuenta con su grupo, segmento, canal y subcanal.

In [2]:
ws3 = wb["Planilha3"]
filas = list(ws3.iter_rows(values_only=True))
maestro = pd.DataFrame(filas[1:], columns=filas[0]).dropna(how="all")
maestro = maestro[maestro["COD KEY ACCOUNT"].notna()].reset_index(drop=True)
print(f"{len(maestro):,} cuentas".replace(",", "."))
maestro.head(10)

1.105 cuentas

,COD KEY ACCOUNT,Key account,GRUPO KEY ACCOUNT,SEGMENTO CUSTOMER,Canal,Subcanal
0,1041.0,SM BARBOSA,SM BARBOSA,REGIONAIS NORTE,SUPERMERCADO 5 -19 CKS,HIPER & SUPER
1,5033.0,REG-BIG MAIS,REG-BIG MAIS,SEM CUSTOMER,SUPERMERCADO 5 -19 CKS,HIPER & SUPER
2,177.0,MAXXI,BIG ATACADO,SEM CUSTOMER,ATACADISTA,INDIRETOS
3,7465.0,CESTTO,CESTTO,SEM CUSTOMER,ATACADISTA,INDIRETOS
4,7463.0,LEVE MAIS ATACADISTA,ASUN,REGIONAIS SUL,ATACADISTA,INDIRETOS
5,7122.0,SUPER SCHIMIT,SUPER SCHIMIT,REGIONAIS SUL,ATACADISTA,INDIRETOS
6,7466.0,COOPER ATACADO,COOPER,REGIONAIS SUL,ATACADISTA,INDIRETOS
7,2480.0,CASH ATACADO,CASH ATACADO,SEM CUSTOMER,ATACADISTA,INDIRETOS
8,185.0,BERGAMINI,BERGAMINI,SEM CUSTOMER,HIPERMERCADO,HIPER & SUPER
9,1136.0,MAGAZINE LUIZA,DIGITAL,SEM CUSTOMER,SUPERMERCADO 5 -19 CKS,HIPER & SUPER


In [3]:
# Resumen del maestro: cuentas por SEGMENTO y por Canal
print("Cuentas por SEGMENTO CUSTOMER:")
print(maestro["SEGMENTO CUSTOMER"].value_counts())
print("\nCuentas por Canal:")
print(maestro["Canal"].value_counts())

Cuentas por SEGMENTO CUSTOMER:
SEGMENTO CUSTOMER
-                  706
SEM CUSTOMER       258
REGIONAIS SUL       36
ON PREMISE          35
REGIONAIS NORTE     33
NACIONAIS           24
ATACADOS             8
PROXIMIDADE          3
DIAMANTE             2
Name: count, dtype: int64

Cuentas por Canal:
Canal
-                             226
SUPERMERCADO 5 -19 CKS        139
FAST FOOD                     116
RESTAURANTE PREMIUM           115
ATACADISTA                     55
CAFETERIA                      50
PRATOS RAPIDOS                 41
MERCADO AUTONOMO               36
SANDUICHERIA                   34
LOJA DE CONVENIÊNCIA           28
DROGARIA                       25
RESTAURANTE                    22
SUPERMERCADO 20 - 49 CKS       21
LOJA DE DEPARTAMENTO           21
CINEMA/TEATRO                  20
COZINHA INDUSTRIAL             18
BAR NOTURNO/CHOPPERIA          18
HOTEL                          15
COMPRA CENTRALIZADA            11
MINI MERCADO 3-4               11
NÃO CONVENCI

## Paso 3 · `Planilha2` — extraer las 16 tablas dinámicas

Cada dinámica vive en una ubicación fija de la hoja. Leemos esas celdas
(valores ya calculados) y las convertimos en tablas ordenadas.
Todas miden **`Soma de VOL_CF`** por **STATUS_PALLET_LASTRO**
(`LASTRO FECHADO` · `MISTO` · `PALLET FECHADO` · `LASTRO FRACIONADO`).

In [4]:
# grilla de valores de las primeras 85 filas de Planilha2
ws2 = wb["Planilha2"]
grid = {}
for i, row in enumerate(ws2.iter_rows(min_row=1, max_row=85, values_only=True), start=1):
    for j, v in enumerate(row, start=1):
        if v is not None:
            grid[(i, j)] = v

def limpiar(s):
    if not isinstance(s, str): return s
    return (s.replace("\ufffd", "o").replace("R\ufffdtulos", "Rotulos").strip())

def extraer(ref, dimension):
    c1, r1, c2, r2 = range_boundaries(ref)
    blk = [[grid.get((rr, cc)) for cc in range(c1, c2+1)] for rr in range(r1, r2+1)]
    hdr = blk[1]                                   # fila 'Rotulos de Linha | LASTRO FECHADO | ...'
    cols = [dimension] + [limpiar(h) for h in hdr[1:]]
    data = [r for r in blk[2:] if r[0] is not None]
    df = pd.DataFrame(data, columns=cols).set_index(dimension)
    return df

# Mapa de las 16 dinamicas: ubicacion -> dimension de fila
PIVOTS = [
    ("B9:F21",   "SEGMENTO CUSTOMER"), ("I9:M21",   "SEGMENTO CUSTOMER"),
    ("Q9:U15",   "Canal"), ("W9:AA15",  "Canal"), ("Q20:U33", "Canal"),
    ("W20:AA33", "Canal"), ("Q38:U45",  "Canal"), ("W38:AA45","Canal"), ("W74:AA81","Canal"),
    ("AE9:AI16", "GRUPO KEY ACCOUNT"), ("AK9:AO16","GRUPO KEY ACCOUNT"),
    ("AE21:AI28","GRUPO KEY ACCOUNT"), ("AK21:AO28","GRUPO KEY ACCOUNT"),
    ("AE35:AI42","Key account"), ("AK35:AO42","Key account"), ("B26:F36","Key account / Subcanal"),
]
def es_pct(df):
    tg = [c for c in df.columns if "Total" in str(c)]
    if not tg: return False
    v = pd.to_numeric(df[tg[0]], errors="coerce").dropna()
    return len(v)>0 and (v.round(3)==1).mean() > 0.8

TAB = []                          # lista de dinamicas: (dimension, tipo, DataFrame)
for ref,dim in PIVOTS:
    t = extraer(ref, dim)
    TAB.append((dim, "%" if es_pct(t) else "abs", t))
print(f"{len(TAB)} dinamicas extraidas.")

def pick(dim, kind):
    """Devuelve la 1a dinamica que coincide con (dimension, tipo)."""
    for d,k,t in TAB:
        if d==dim and k==kind: return t
    return None

pd.DataFrame([(d,k,t.shape[0],list(t.columns)) for d,k,t in TAB],
             columns=["dimension","tipo","filas","columnas"])

16 dinamicas extraidas.


,dimension,tipo,filas,columnas
0,SEGMENTO CUSTOMER,abs,11,"[LASTRO FECHADO, MISTO, PALLET FECHADO, Total ..."
1,SEGMENTO CUSTOMER,%,11,"[LASTRO FECHADO, MISTO, PALLET FECHADO, Total ..."
2,Canal,%,5,"[LASTRO FECHADO, MISTO, PALLET FECHADO, Total ..."
3,Canal,abs,5,"[LASTRO FECHADO, MISTO, PALLET FECHADO, Total ..."
4,Canal,%,12,"[LASTRO FECHADO, MISTO, PALLET FECHADO, Total ..."
5,Canal,abs,12,"[LASTRO FECHADO, LASTRO FRACIONADO, PALLET FEC..."
6,Canal,%,6,"[LASTRO FECHADO, MISTO, PALLET FECHADO, Total ..."
7,Canal,abs,6,"[LASTRO FECHADO, LASTRO FRACIONADO, PALLET FEC..."
8,Canal,%,6,"[LASTRO FECHADO, LASTRO FRACIONADO, PALLET FEC..."
9,GRUPO KEY ACCOUNT,%,6,"[LASTRO FECHADO, MISTO, PALLET FECHADO, Total ..."


### 3a · Por SEGMENTO CUSTOMER  — absoluto  &nbsp;*(Total Geral = 73.981.990)*

In [5]:
fmt   = lambda df: df.style.format(lambda v: f"{v:,.0f}".replace(",",".") if isinstance(v,(int,float)) else v)
heatp = lambda df: df.style.format(lambda v: f"{v:.1%}" if isinstance(v,(int,float)) else v).background_gradient(cmap="RdYlGn", axis=None)
fmt(pick("SEGMENTO CUSTOMER","abs"))

,LASTRO FECHADO,MISTO,PALLET FECHADO,Total Geral
SEGMENTO CUSTOMER,,,,
REGIONAIS NORTE,2.086.380,9.012.101,3.171.130,14.269.611
SEM CUSTOMER,1.862.479,8.470.920,3.368.989,13.702.388
REGIONAIS SUL,2.899.876,7.217.835,5.314.034,15.431.745
NACIONAIS,430.803,4.364.846,1.808.225,6.603.874
ATACADOS,2.531.048,4.201.193,7.911.417,14.643.658
-,130.576,3.861.485,1.093.240,5.085.301
ON PREMISE,62.664,1.869.610,119.832,2.052.106
PROXIMIDADE,44.385,1.238.015,442.470,1.724.870
DIAMANTE,28.918,216.331,220.885,466.134


### 3b · Por SEGMENTO CUSTOMER — % de armado  *(heatmap)*

In [6]:
heatp(pick("SEGMENTO CUSTOMER","%"))

,LASTRO FECHADO,MISTO,PALLET FECHADO,Total Geral
SEGMENTO CUSTOMER,,,,
REGIONAIS NORTE,14.6%,63.2%,22.2%,100.0%
SEM CUSTOMER,13.6%,61.8%,24.6%,100.0%
REGIONAIS SUL,18.8%,46.8%,34.4%,100.0%
NACIONAIS,6.5%,66.1%,27.4%,100.0%
ATACADOS,17.3%,28.7%,54.0%,100.0%
-,2.6%,75.9%,21.5%,100.0%
ON PREMISE,3.1%,91.1%,5.8%,100.0%
PROXIMIDADE,2.6%,71.8%,25.7%,100.0%
DIAMANTE,6.2%,46.4%,47.4%,100.0%


### 3c · Por Canal — absoluto

In [7]:
fmt(pick("Canal","abs"))

,LASTRO FECHADO,MISTO,PALLET FECHADO,Total Geral
Canal,,,,
SUPERMERCADO 5 -19 CKS,1.159.388,5.391.403,1.396.439,7.947.230
ATACADISTA,527.731,1.761.259,1.088.402,3.377.392
SUPERMERCADO 20 - 49 CKS,364.381,1.756.741,547.105,2.668.227
COMPRA CENTRALIZADA,34.880,102.698,139.184,276.762
Total Geral,2.086.380,9.012.101,3.171.130,14.269.611


### 3c' · Por Canal — % de armado *(heatmap)*

In [8]:
heatp(pick("Canal","%"))

,LASTRO FECHADO,MISTO,PALLET FECHADO,Total Geral
Canal,,,,
SUPERMERCADO 5 -19 CKS,14.6%,67.8%,17.6%,100.0%
ATACADISTA,15.6%,52.1%,32.2%,100.0%
SUPERMERCADO 20 - 49 CKS,13.7%,65.8%,20.5%,100.0%
COMPRA CENTRALIZADA,12.6%,37.1%,50.3%,100.0%
Total Geral,14.6%,63.2%,22.2%,100.0%


### 3d · Por GRUPO KEY ACCOUNT — absoluto y %

In [9]:
display(fmt(pick("GRUPO KEY ACCOUNT","abs")))
heatp(pick("GRUPO KEY ACCOUNT","%"))

,LASTRO FECHADO,MISTO,PALLET FECHADO,Total Geral
GRUPO KEY ACCOUNT,,,,
BH SUPERMERCADOS,565.367,1.823.629,553.810,2.942.806
EPA,64.700,737.475,54.159,856.334
VILLEFORT,51.084,721.221,181.031,953.336
SONDA,234.325,592.573,209.074,1.035.972
MART MINAS,343.594,503.899,516.268,1.363.761
Total Geral,1.259.070,4.378.797,1.514.342,7.152.209


,LASTRO FECHADO,MISTO,PALLET FECHADO,Total Geral
GRUPO KEY ACCOUNT,,,,
BH SUPERMERCADOS,19.2%,62.0%,18.8%,100.0%
EPA,7.6%,86.1%,6.3%,100.0%
VILLEFORT,5.4%,75.7%,19.0%,100.0%
SONDA,22.6%,57.2%,20.2%,100.0%
MART MINAS,25.2%,36.9%,37.9%,100.0%
Total Geral,17.6%,61.2%,21.2%,100.0%


### 3e · Por Key account — absoluto y %

In [10]:
display(fmt(pick("Key account","abs")))
heatp(pick("Key account","%"))

,LASTRO FECHADO,LASTRO FRACIONADO,PALLET FECHADO,Total Geral
Key account,,,,
CARREFOUR HIPER,93.122,1.557.760,129.203,1.780.085
PAO DE ACUCAR,47.784,1.097.070,57.634,1.202.488
DAKI STORE,5.904,177.356,73.418,256.678
AVOCADO,4.531,133.509,18.058,156.098
AMAZON,133.758,19.042,73.661,226.461
Total Geral,285.099,2.984.737,351.974,3.621.810


,LASTRO FECHADO,LASTRO FRACIONADO,PALLET FECHADO,Total Geral
Key account,,,,
CARREFOUR HIPER,5.2%,87.5%,7.3%,100.0%
PAO DE ACUCAR,4.0%,91.2%,4.8%,100.0%
DAKI STORE,2.3%,69.1%,28.6%,100.0%
AVOCADO,2.9%,85.5%,11.6%,100.0%
AMAZON,59.1%,8.4%,32.5%,100.0%
Total Geral,7.9%,82.4%,9.7%,100.0%


## Paso 4 · Exportar todo a un Excel limpio
Una hoja para el maestro y una hoja por cada dinámica extraída.

In [11]:
SALIDA = Path.cwd() / "Paletizacao_extraido.xlsx"
cont = {}
with pd.ExcelWriter(SALIDA, engine="openpyxl") as xw:
    maestro.to_excel(xw, sheet_name="Maestro_KeyAccount", index=False)
    for i,(dim,kind,t) in enumerate(TAB, start=1):
        nombre = f"P2_{i:02d}_{dim.split()[0]}_{kind}"[:31]
        t.to_excel(xw, sheet_name=nombre)
print("Exportado ->", SALIDA.name, "· hojas:", 1+len(TAB))

Exportado -> Paletizacao_extraido.xlsx · hojas: 17


## Resumen

- **`Planilha3`** extraída completa: maestro de **1.105 Key Accounts**
  (COD · Key account · GRUPO · SEGMENTO · Canal · Subcanal).
- **`Planilha2`** extraída: **16 dinámicas** de `Soma de VOL_CF` por tipo de armado,
  cruzadas por SEGMENTO / Canal / GRUPO / Key account (absoluto y %).
- **No se reconstruye** desde datos: la fuente es Power Query externo y es otro
  universo (73.98 M cajas). Esto es una **extracción** fiel de lo ya calculado.
- Todo exportado a `Paletizacao_extraido.xlsx`.